# A County-Level Suitability Analysis for New Data Center Sites in the United States

In [7]:
import arcpy
import os
import pandas as pd

arcpy.env.parallelProcessingFactor = "0" 
arcpy.env.scratchWorkspace = "memory"
conus_albers_sr = arcpy.SpatialReference(5070)
print(arcpy.CheckOutExtension("Spatial"))
arcpy.env.overwriteOutput = True

from arcpy.sa import *

data_dir = r".\data"

temp_gdb = os.path.join(data_dir, "temp.gdb")
if not arcpy.Exists(temp_gdb):
    arcpy.management.CreateFileGDB(data_dir, "temp.gdb")

arcpy.env.workspace = data_dir
arcpy.env.scratchWorkspace = temp_gdb

CheckedOut


In [10]:
counties_shp = os.path.join(data_dir, "US_Contig_County_Boundaries", "US_County_Boundaries.shp")
master_fc = os.path.join(temp_gdb, "Counties_Suitability_Master")
if not arcpy.Exists(master_fc):
    arcpy.management.CopyFeatures(counties_shp, master_fc)
    print(" -> Master feature class created.")

## 1. Dissolve state features into single contiguous feature

In [2]:
input_states = os.path.join(data_dir, "US_Contig_State_Boundaries", "US_Contig_State_Boundaries.shp")
us_contig_boundary_dir = os.path.join(data_dir, "US_Contiguous_Boundary")
contiguous_us = os.path.join(us_contig_boundary_dir, "US_Contiguous_Boundary.shp")
os.makedirs(us_contig_boundary_dir, exist_ok=True)

print("Dissolving 48 state boundaries into a single contiguous feature...")

arcpy.management.Dissolve(
    in_features=input_states,
    out_feature_class=contiguous_us
)

print(f"Success! Master boundary saved to: {contiguous_us}")

Dissolving 48 state boundaries into a single contiguous feature...
Success! Master boundary saved to: .\data\US_Contiguous_Boundary\US_Contiguous_Boundary.shp


## 2. Reproject Boundary to EPSG:5070 (NAD83 / Conus Albers)

In [3]:
projected_contiguous_us = os.path.join(us_contig_boundary_dir, "US_Contiguous_Boundary_5070.shp")

print("Projecting boundary to EPSG:5070...")

arcpy.management.Project(
    in_dataset=contiguous_us,
    out_dataset=projected_contiguous_us,
    out_coor_system=conus_albers_sr
)

print(f"Success! Projected boundary saved to: {projected_contiguous_us}")

Projecting boundary to EPSG:5070...
Success! Projected boundary saved to: .\data\US_Contiguous_Boundary\US_Contiguous_Boundary_5070.shp


## 3. Slope Raster
Clip and standardize slope raster

In [4]:
raw_slope_tif = os.path.join(data_dir, "slope_1KMmd_GMTEDmd.tif")
slope_processing_dir = os.path.join(data_dir, "Slope_Processing")
os.makedirs(slope_processing_dir, exist_ok=True)

clipped_slope_tif = os.path.join(slope_processing_dir, "Slope_Clipped_5070.tif")
standardized_slope_tif = os.path.join(slope_processing_dir, "Slope_Standardized.tif")

print("Extracting slope raster using the contiguous US boundary mask...")

with arcpy.EnvManager(outputCoordinateSystem=conus_albers_sr, snapRaster=raw_slope_tif):
    clipped_slope = ExtractByMask(
        in_raster=raw_slope_tif,
        in_mask_data=projected_contiguous_us
    )
    clipped_slope.save(clipped_slope_tif)

print("Excluding slopes > 10 degrees and scaling remaining areas from 0.0 to 1.0...")

# Map Algebra logic:
# 1. 'clipped_slope <= 10' acts as a conditional filter. Anything > 10 becomes NoData.
# 2. '1.0 - (clipped_slope / 10.0)' inverts and normalizes the scale.
#    A slope of 0 degrees becomes 1.0 (highly suitable).
#    A slope of 10 degrees becomes 0.0 (least suitable).

standardized_slope = Con(clipped_slope <= 10, 1.0 - (clipped_slope / 10.0))
standardized_slope.save(standardized_slope_tif)

print(f"Success! Standardized slope raster saved to: {standardized_slope_tif}")

Extracting slope raster using the contiguous US boundary mask...
Excluding slopes > 10 degrees and scaling remaining areas from 0.0 to 1.0...
Success! Standardized slope raster saved to: .\data\Slope_Processing\Slope_Standardized.tif


## 4. Proximity to Water Raster
Generate and standardize distance to water surface

Note to future selves: a limitation of this is that it does not account for underground water sources nor the volume of water, so it's flawed

In [5]:
"""
lakes_shp = os.path.join(data_dir, "rivers_and_lakes_shapefile", "NA_Lakes_and_Rivers", "data", "lakes_p", "northamerica_lakes_cec_2023.shp")
rivers_shp = os.path.join(data_dir, "rivers_and_lakes_shapefile", "NA_Lakes_and_Rivers", "data", "rivers_l", "northamerica_rivers_cec_2023.shp")

water_processing_dir = os.path.join(data_dir, "Water_Processing")
os.makedirs(water_processing_dir, exist_ok=True)

dist_water_tif = os.path.join(water_processing_dir, "Distance_to_Water_5070.tif")
standardized_water_tif = os.path.join(water_processing_dir, "Distance_to_Water_Standardized.tif")

print("Calculating distance to lakes and rivers within the US boundary...")

with arcpy.EnvManager(extent=projected_contiguous_us, mask=projected_contiguous_us, outputCoordinateSystem=conus_albers_sr, cellSize=1000):    
    dist_lakes = EucDistance(lakes_shp)    
    dist_rivers = EucDistance(rivers_shp)
    
    min_dist_water = CellStatistics([dist_lakes, dist_rivers], "MINIMUM", "DATA")
    min_dist_water.save(dist_water_tif)

print("Standardizing water distance to a 0.0 to 1.0 suitability scale...")

max_dist = float(arcpy.management.GetRasterProperties(min_dist_water, "MAXIMUM").getOutput(0))

# Invert and normalize the scale: 
# A distance of 0 becomes 1.0 (highly suitable), the maximum distance becomes 0.0
standardized_water = 1.0 - (min_dist_water / max_dist)
standardized_water.save(standardized_water_tif)

print(f"Success! Standardized water raster saved to: {standardized_water_tif}")
"""
print("this is commented out because its dumb")

this is commented out because its dumb


## 5. Climate / Temperature Processing
Join aggregated county temperatures and standardize to a suitability raster.

In [11]:
print("Step 1: Processing NOAA temperatures using Pandas...")
avg_temp_csv = os.path.join(data_dir, "avg_temp_counties.csv")
df = pd.read_csv(avg_temp_csv)

state_fips = {
    'Alabama': '01', 'Arizona': '04', 'Arkansas': '05', 'California': '06',
    'Colorado': '08', 'Connecticut': '09', 'Delaware': '10', 'District of Columbia': '11', 
    'Florida': '12', 'Georgia': '13', 'Idaho': '16', 'Illinois': '17', 'Indiana': '18',
    'Iowa': '19', 'Kansas': '20', 'Kentucky': '21', 'Louisiana': '22',
    'Maine': '23', 'Maryland': '24', 'Massachusetts': '25', 'Michigan': '26',
    'Minnesota': '27', 'Mississippi': '28', 'Missouri': '29', 'Montana': '30',
    'Nebraska': '31', 'Nevada': '32', 'New Hampshire': '33', 'New Jersey': '34',
    'New Mexico': '35', 'New York': '36', 'North Carolina': '37', 'North Dakota': '38',
    'Ohio': '39', 'Oklahoma': '40', 'Oregon': '41', 'Pennsylvania': '42',
    'Rhode Island': '44', 'South Carolina': '45', 'South Dakota': '46',
    'Tennessee': '47', 'Texas': '48', 'Utah': '49', 'Vermont': '50',
    'Virginia': '51', 'Washington': '53', 'West Virginia': '54', 'Wisconsin': '55',
    'Wyoming': '56'
}

# Generate standard 5-digit GEOIDs
df['GEOID'] = df['State'].map(state_fips) + df['ID'].str.split('-').str[1]
temp_dict = dict(zip(df['GEOID'], df['Value']))

# Calculate min and max directly in Python to bypass ArcPy statistics tools
min_temp = df['Value'].min()
max_temp = df['Value'].max()

print("Step 2: Applying the Virginia Independent City Patch...")
# This maps the "holes" (Independent Cities) to their surrounding county GEOID
# Example: Alexandria (51510) & Fairfax City (51600) get the temperature of Fairfax County (51059)
va_patch = {
    "51510": "51059", "51600": "51059", "51610": "51059", # Fairfax County surrounds
    "51540": "51003", # Charlottesville -> Albemarle
    "51760": "51087", # Richmond City -> Henrico
    "51770": "51161", # Roanoke City -> Roanoke County
    "51810": "51163", # Virginia Beach -> Rockbridge (proxy)
    "11001": "24031"  # Washington DC -> Montgomery County, MD
}

# Update the main dictionary with the patched values
for city_geoid, county_geoid in va_patch.items():
    if county_geoid in temp_dict:
        temp_dict[city_geoid] = temp_dict[county_geoid]

print("Step 3: Injecting data and standardizing scores in the attribute table...")
# Add columns for this specific criteria
arcpy.management.AddField(master_fc, "Temp_Raw", "DOUBLE")
arcpy.management.AddField(master_fc, "Temp_Score", "DOUBLE")

# Use UpdateCursor to write the values directly into the Master Shapefile
with arcpy.da.UpdateCursor(master_fc, ["GEOID", "Temp_Raw", "Temp_Score"]) as cursor:
    for row in cursor:
        geoid = str(row[0]).zfill(5) 
        
        if geoid in temp_dict:
            raw_val = float(temp_dict[geoid])
            row[1] = raw_val
            
            # Standardization math: 1.0 - ((Value - Min) / (Max - Min))
            # Lower temperatures = Score closer to 1.0
            row[2] = 1.0 - ((raw_val - min_temp) / (max_temp - min_temp))
        else:
            # Leave Null if data is missing
            row[1] = None
            row[2] = None
            
        cursor.updateRow(row)

# Reset workspace
arcpy.env.workspace = data_dir

print("Success! Temperature values and standardized scores added to Counties_Suitability_Master.")

Step 1: Processing NOAA temperatures using Pandas...
Step 2: Applying the Virginia Independent City Patch...
Step 3: Injecting data and standardizing scores in the attribute table...
Success! Temperature values and standardized scores added to Counties_Suitability_Master.
